# Análise de Engajamento e Monetização na Steam
**Autor:** Danilo Rodrigues Guerreiro

**Objetivo:** Analisar o tempo médio de jogo, volume e taxa de aprovação dos gêneros na plataforma Steam, tratando dados nulos e removendo categorias utilitárias que não são jogos.

In [29]:
import pandas as pd

df = pd.read_csv('games.csv')


## 1. Seleção de Colunas e Limpeza de Dados
Nesta etapa, selecionamos apenas as colunas relevantes para a análise de engajamento e criamos a métrica totalizadora de avaliações. Removemos registros nulos e jogos sem nenhuma avaliação para evitar divisões por zero e dados incorretos.

In [36]:
# Tratamento de Dados
df_analise = df[['AppID', 'Genres', 'Price', 'Average playtime forever', 'Positive', 'Negative']]

df_analise['Total_Avaliacoes'] = df_analise['Positive'] + df_analise['Negative']

df_analise = df_analise.dropna()
df_analise = df_analise[df_analise['Total_Avaliacoes'] > 0]

display(df_analise.head())

,AppID,Genres,Price,Average playtime forever,Positive,Negative,Total_Avaliacoes
496350,Supipara - Chapter 1 Spring Has Come!,Adventure,65,8,252,3,255
1034400,Mystery Solitaire The Black Raven,Casual,0,0,21,3,24
1934300,Armored Brigade II,"Simulation,Strategy",10,675,117,13,130
1157670,Hepta Beats,"Casual,Indie",0,0,28,4,32
1540330,MUMBA IV: Egypt Jewels,"Action,Casual,Indie,Strategy",0,0,19,6,25


## 2. Tratamento de Gêneros Híbridos
Como muitos jogos na Steam possuem múltiplos gêneros na mesma célula (ex: "Action, RPG"), utilizamos a função .explode() para desmembrar as listas e garantir que cada gênero seja contabilizado individualmente.

In [39]:
# Tratar Generos Hibridos

df_analise['Genero_Lista'] = df_analise['Genres'].str.split(',')

df_analise = df_analise.explode('Genero_Lista')

df_analise['Genero_Lista'] = df_analise['Genero_Lista'].str.strip()

display(df_analise.head())

,AppID,Genres,Price,Average playtime forever,Positive,Negative,Total_Avaliacoes,Genero_Lista
496350,Supipara - Chapter 1 Spring Has Come!,Adventure,65,8,252,3,255,Adventure
1034400,Mystery Solitaire The Black Raven,Casual,0,0,21,3,24,Casual
1934300,Armored Brigade II,"Simulation,Strategy",10,675,117,13,130,Simulation
1934300,Armored Brigade II,"Simulation,Strategy",10,675,117,13,130,Strategy
1934300,Armored Brigade II,"Simulation,Strategy",10,675,117,13,130,Simulation


## 3. Filtragem de Categorias Utilitárias e Agrupamento
A base de dados contém softwares e utilitários (ex: "Audio Production", "Utilities") que distorcem o tempo médio de jogo. Removemos essas categorias não relacionadas a jogos e agrupamos os dados para calcular o engajamento médio e a taxa de aprovação.

In [47]:
# Agrupar e Calcular as Métricas

softwares_utilitarios = [
    'Audio Production', 'Web Publishing', 'Video Production', 
    'Utilities', 'Software Training', 'Animation & Modeling', 
    'Design & Illustration', 'Photo Editing', 'Education', 'Game Development','Massively Multiplayer'
]

df_apenas_jogos = df_analise[~df_analise['Genero_Lista'].isin(softwares_utilitarios)].copy()

top_generos = df_apenas_jogos.groupby('Genero_Lista').agg(
    Qtd_Jogos=('AppID', 'count'),
    Tempo_Medio_Minuto=('Average playtime forever', 'mean'),
    Total_Positivas=('Positive', 'sum'),
    Total_Avaliacoes=('Total_Avaliacoes', 'sum')
)

top_generos['Taxa_Aprovacao_Pct'] = (top_generos['Total_Positivas'] / top_generos['Total_Avaliacoes']) * 100
top_generos['Taxa_Aprovacao_Pct'] = top_generos['Taxa_Aprovacao_Pct'].round(2)

top_generos['Tempo_Medio_Minuto'] = top_generos['Tempo_Medio_Minuto'].round(0)

top_10_jogos = top_generos.sort_values(by='Tempo_Medio_Minuto', ascending=False).head(10)

display(top_10_jogos)

top_10_jogos.to_csv('top_10_generos_steam.csv', index=False)

,Qtd_Jogos,Tempo_Medio_Minuto,Total_Positivas,Total_Avaliacoes,Taxa_Aprovacao_Pct
Genero_Lista,,,,,
Free To Play,18315,1645.0,81372015,102741803,79.20
Racing,12403,1258.0,12645145,14681187,86.13
RPG,57419,490.0,113200481,131719205,85.94
Simulation,63723,328.0,92996694,107757363,86.30
Casual,113669,305.0,68559257,78307200,87.55
Strategy,60269,300.0,71951403,84201062,85.45
Indie,190284,285.0,184350637,207983665,88.64
Adventure,114990,265.0,171830273,203527440,84.43
Sports,15190,261.0,14898351,18408725,80.93


## 4. Conclusão dos Resultados
- **Top Gêneros por Engajamento:** Categoria x, y e z lideram em tempo médio de jogo.

- **Taxa de Aprovação:** A aprovação média dos principais gêneros se mantém acima de 80%.

- **Tratamento de Dados:** A remoção de softwares utilitários garantiu que a média de tempo de jogo refletisse apenas a experiência de jogos reais na plataforma.